# Partition Cell Types for Clustering and Harmonization 

To ensure that our cell type labels are as accurate as possible, we'll subset our dataset based on our `AIFI_L3` labels for harmonization and clustering


## Load libraries

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import scanpy.external as sce

In [2]:
out_dir = 'unharmonized_output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [3]:
### slice indices for L3 cell types
start = 0
end = 20

## Helper functions

These make it a bit simpler to cache and read in files from HISE

In [4]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [5]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

In [6]:
def sort_adata_uuid(uuid, sort_cols = ['AIFI_L2', 'sample.sampleKitGuid']):
    cache_file = cache_uuid_path(uuid)
    adata = sc.read_h5ad(cache_file)
    obs = adata.obs
    obs = obs.sort_values(sort_cols)
    adata = adata[obs.index]
    adata.write_h5ad(cache_file)

This function will enable us to connect to our .h5ad files without loading the entire thing into memory. We'll then load only the cells that we want for each cell class to assemble them for writing. This should save us some overhead as we do our subsetting.

In [7]:
def read_adata_backed_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file, backed = 'r')
    return res

This function will remove Ig-related genes, which is recommended by Marla Glass for analysis of B cell subtypes

In [8]:
def remove_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes

    filtered_genes = [gene for gene in adata.var_names if gene not in exl_genes]
    adata = adata[:, filtered_genes]

    return adata

This function will apply a standard normalization, nearest neighbors, clustering, and UMAP process to our cell subsets:

In [9]:
def process_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    #adata = adata.raw.to_adata()  #temi - took out for unharmonization
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')
    
    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_{r}'.format(r = resolution),
        n_iterations = 2
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05)

    print('Ranking genes', end = "; ")
    sc.tl.rank_genes_groups(adata, 'leiden_{r}'.format(r = resolution), method='t-test')
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [10]:
def process_harmonize_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    #adata = adata.raw.to_adata()
    #adata.raw = adata
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')


    # Perform the operations on the Anndata object
    print('Harmonize', end = "; ")
    sce.pp.harmony_integrate(adata, ["batch_id", "subject.subjectGuid"])

    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_unharmony_{r}'.format(r = resolution),
        n_iterations = 2,
        #key_added="leiden_harmony"
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05, init_pos="X_pca_harmony")

    print('Ranking genes', end = "; ")
    sc.tl.rank_genes_groups(adata, 'leiden_ harmony_{r}'.format(r = resolution), method='t-test')
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [11]:
def format_cell_type(cell_type):
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [12]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Identify files for use in HISE

In [13]:
search_id = 'hafnium-copper-praseodymium'

Retrieve files stored in our HISE project store

In [14]:
ps_df = hisepy.list_files_in_project_store('Dyna_IHandA')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [15]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [16]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [17]:
h5ad_df

,id,name
68,7e87a3d2-3ac0-4b16-98e9-3d75753d128c,hafnium-copper-praseodymium/pbmc_set1_initial_...
71,cdcfd299-0f57-4b74-843a-fa57c157be15,hafnium-copper-praseodymium/pbmc_set2_initial_...
148,8d650819-d180-4174-8d6a-1741bee03a65,hafnium-copper-praseodymium/up1_cluster_unharm...
149,a6b14649-a818-4be1-b4e1-33fa9699191c,hafnium-copper-praseodymium/up1_cluster_unharm...
150,89a5f43e-b12b-457f-b773-c142734c6ffa,hafnium-copper-praseodymium/up1_cluster_unharm...
151,9ea97469-c633-4d1c-ace0-7b05b1033ab9,hafnium-copper-praseodymium/up1_cluster_unharm...
152,2fa96676-8195-416c-9121-fb6c29b03ef1,hafnium-copper-praseodymium/up1_cluster_unharm...
153,0d410e84-9979-4212-9a91-8fb6ab8d53f4,hafnium-copper-praseodymium/up1_cluster_unharm...
154,9c499934-90bf-47a0-86d9-2ae4d9da395e,hafnium-copper-praseodymium/up1_cluster_unharm...
155,2b1c360f-38d5-4769-a05b-24c6792e3571,hafnium-copper-praseodymium/up1_cluster_unharm...


In [21]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]
    #print(group_name)

set1
set2
hafnium-copper-praseodymium/up1_cluster_unharmonize_ASDC_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_Activated_memory_B_cell_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_Adaptive_NK_cell_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_BaEoMaP_cell_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_C1Qpos_CD16_monocyte_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD27neg_effector_B_cell_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD27pos_effector_B_cell_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD4_MAIT_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD56bright_NK_cell_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD8_MAIT_2024-09-30.h5ad
hafnium-copper-praseodymium/up1_cluster_unharmonize_CD8aa_2024-0

In [22]:
type(group_name)

str

## Download and sort .h5ad files

Sorting the .h5ad files by AIFI_L2 will make reading each cell type much faster by placing cells of the same type next to each other in the sparse matrix.

In [19]:
##### ONLY NEED TO RUN ONCE

for uuid in h5ad_uuids.values():
    sort_adata_uuid(uuid, sort_cols = ['AIFI_L3'])

downloading fileID: 7e87a3d2-3ac0-4b16-98e9-3d75753d128c
Files have been successfully downloaded!
downloading fileID: cdcfd299-0f57-4b74-843a-fa57c157be15
Files have been successfully downloaded!


## Open connections to .h5ad files

Now that they're sorted, we can open these files with on-disk backing so we don't have to read the entire file at once.

In [20]:
h5ad_conn = {}
for group_name, uuid in h5ad_uuids.items():
    h5ad_conn[group_name] = read_adata_backed_uuid(uuid)

## Process each cell type

In [21]:
adata = read_adata_backed_uuid(list(h5ad_uuids.values())[0])
l3_types = adata.obs['AIFI_L3'].unique().tolist()
l3_types.sort()

In [22]:
h5ad_conn

{'set1': AnnData object with n_obs × n_vars = 1851815 × 33538 backed at '/home/jupyter/cache/7e87a3d2-3ac0-4b16-98e9-3d75753d128c/pbmc_set1_initial_qc_2024-08-19.h5ad'
     obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_c

In [23]:
l3_types

['ASDC',
 'Activated memory B cell',
 'Adaptive NK cell',
 'BaEoMaP cell',
 'C1Q+ CD16 monocyte',
 'CD14+ cDC2',
 'CD27+ effector B cell',
 'CD27- effector B cell',
 'CD4 MAIT',
 'CD56bright NK cell',
 'CD8 MAIT',
 'CD8aa',
 'CD95 memory B cell',
 'CLP cell',
 'CM CD4 T cell',
 'CM CD8 T cell',
 'CMP cell',
 'Core CD14 monocyte',
 'Core CD16 monocyte',
 'Core memory B cell',
 'Core naive B cell',
 'Core naive CD4 T cell',
 'Core naive CD8 T cell',
 'DN T cell',
 'Early memory B cell',
 'Erythrocyte',
 'GZMB+ Vd2 gdT',
 'GZMB- CD27+ EM CD4 T cell',
 'GZMB- CD27- EM CD4 T cell',
 'GZMK+ CD27+ EM CD8 T cell',
 'GZMK+ CD56dim NK cell',
 'GZMK+ Vd2 gdT',
 'GZMK+ memory CD4 Treg',
 'GZMK- CD27+ EM CD8 T cell',
 'GZMK- CD56dim NK cell',
 'HLA-DRhi cDC2',
 'IL1B+ CD14 monocyte',
 'ILC',
 'ISG+ CD14 monocyte',
 'ISG+ CD16 monocyte',
 'ISG+ CD56dim NK cell',
 'ISG+ MAIT',
 'ISG+ cDC2',
 'ISG+ memory CD4 T cell',
 'ISG+ memory CD8 T cell',
 'ISG+ naive B cell',
 'ISG+ naive CD4 T cell',
 'ISG+ na

In [24]:
l3_types_sub = l3_types[start:end]

l3_types_sub

['ASDC',
 'Activated memory B cell',
 'Adaptive NK cell',
 'BaEoMaP cell',
 'C1Q+ CD16 monocyte',
 'CD14+ cDC2',
 'CD27+ effector B cell',
 'CD27- effector B cell',
 'CD4 MAIT',
 'CD56bright NK cell',
 'CD8 MAIT',
 'CD8aa',
 'CD95 memory B cell',
 'CLP cell',
 'CM CD4 T cell',
 'CM CD8 T cell',
 'CMP cell',
 'Core CD14 monocyte',
 'Core CD16 monocyte',
 'Core memory B cell']

In [25]:
def read_l3_type(adata, cell_type):
    type_adata = adata[adata.obs['AIFI_L3'] == cell_type].to_memory()
    return type_adata

In [26]:
out_files = []

### Harmonize

In [29]:
%%time

for cell_type in l3_types_sub:
    print(cell_type)
    
    # Read data from each group for this type in parallel
    print('Loading data')
    type_adata_dict = {}

    with ThreadPoolExecutor(max_workers = 4) as executor:
        futures = {
            executor.submit(
                read_l3_type, 
                h5ad_conn[group_name], 
                cell_type): group_name 
            for group_name in h5ad_conn.keys()
        }
        for future in concurrent.futures.as_completed(futures):
            future_group = futures[future]
            type_adata_dict[future_group] = future.result()
    
    # If small, combine and process
    print('Combining and processing')
    type_adata = sc.concat(type_adata_dict)
    print(type_adata.shape)
    
    if(type_adata.obs['AIFI_L1'][0] == 'B cells'):
        print('Dropping Ig Genes for B cell clustering')
        type_adata = remove_ig_genes(type_adata)
        print(type_adata.shape)
    
    type_adata = process_harmonize_adata(type_adata) # process adata
    
    print('Saving processed data')
    out_type = format_cell_type(cell_type)
    out_file = 'unharmonized_output/up1_cluster_harmonize_{c}_{d}.h5ad'.format(
        g = group_name,
        c = out_type,
        d = date.today()
    )
    type_adata.write_h5ad(out_file)
    out_files.append(out_file)

ASDC
Loading data
Combining and processing
(1301, 33538)
Normalizing; Finding HVGs; Scaling; PCA; 

2024-08-21 19:02:46,901 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


Harmonize; 

2024-08-21 19:02:48,648 - harmonypy - INFO - sklearn.KMeans initialization complete.
2024-08-21 19:02:48,675 - harmonypy - INFO - Iteration 1 of 10
2024-08-21 19:02:49,382 - harmonypy - INFO - Iteration 2 of 10
2024-08-21 19:02:49,911 - harmonypy - INFO - Iteration 3 of 10
2024-08-21 19:02:50,237 - harmonypy - INFO - Iteration 4 of 10
2024-08-21 19:02:50,501 - harmonypy - INFO - Iteration 5 of 10
2024-08-21 19:02:50,762 - harmonypy - INFO - Iteration 6 of 10
2024-08-21 19:02:51,001 - harmonypy - INFO - Iteration 7 of 10
2024-08-21 19:02:51,290 - harmonypy - INFO - Iteration 8 of 10
2024-08-21 19:02:51,585 - harmonypy - INFO - Iteration 9 of 10
2024-08-21 19:02:51,834 - harmonypy - INFO - Iteration 10 of 10
2024-08-21 19:02:52,133 - harmonypy - INFO - Stopped before convergence


Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.
Renormalizing
Saving processed data
Activated memory B cell
Loading data
Combining and processing
(1745, 33538)
Normalizing; Finding HVGs; Scaling; PCA; 

...

Harmonize; 

2024-08-21 21:11:18,102 - harmonypy - INFO - sklearn.KMeans initialization complete.
2024-08-21 21:11:18,594 - harmonypy - INFO - Iteration 1 of 10
2024-08-21 21:12:11,409 - harmonypy - INFO - Iteration 2 of 10
2024-08-21 21:13:02,872 - harmonypy - INFO - Iteration 3 of 10
2024-08-21 21:13:40,784 - harmonypy - INFO - Converged after 3 iterations


Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'scores'] = scores[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

Renormalizing
Saving processed data
CPU times: user 18h 48min 51s, sys: 21h 29min 58s, total: 1d 16h 18min 50s
Wall time: 2h 20min 15s


### Unharmonize

In [33]:
for cell_type in l3_types_sub:
    print(cell_type)
    
    # Read data from each group for this type in parallel
    print('Loading data')
    type_adata_dict = {}

    with ThreadPoolExecutor(max_workers = 4) as executor:
        futures = {
            executor.submit(
                read_l3_type, 
                h5ad_conn[group_name], 
                cell_type): group_name 
            for group_name in h5ad_conn.keys()
        }
        for future in concurrent.futures.as_completed(futures):
            future_group = futures[future]
            type_adata_dict[future_group] = future.result()
    
    # If small, combine and process
    print('Combining and processing')
    type_adata = sc.concat(type_adata_dict)
    print(type_adata.shape)
    
    if(type_adata.obs['AIFI_L1'][0] == 'B cells'):
        print('Dropping Ig Genes for B cell clustering')
        type_adata = remove_ig_genes(type_adata)
        print(type_adata.shape)
    
    type_adata = process_adata(type_adata) # process adata
    
    print('Saving processed data')
    out_type = format_cell_type(cell_type)
    out_file = 'unharmonized_output/up1_cluster_unharmonize_{c}_{d}.h5ad'.format(
        g = group_name,
        c = out_type,
        d = date.today()
    )
    type_adata.write_h5ad(out_file)
    out_files.append(out_file)

ASDC
Loading data
Combining and processing
(1301, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.
Renormalizing
Saving processed data
Activated memory B cell
Loading data
Combining and processing
(1745, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.
Renormalizing
Saving processed data
Adaptive NK cell
Loading data
Combining and processing
(29815, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'scores'] = scores[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

Renormalizing
Saving processed data
BaEoMaP cell
Loading data
Combining and processing
(139, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.
Renormalizing
Saving processed data
C1Q+ CD16 monocyte
Loading data
Combining and processing
(3895, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.
Renormalizing
Saving processed data
CD14+ cDC2
Loading data
Combining and processing
(9288, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'scores'] = scores[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

Renormalizing
Saving processed data
CD27+ effector B cell
Loading data
Combining and processing
(15844, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


...

UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.
UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'scores'] = scores[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

Renormalizing
Saving processed data
Core memory B cell
Loading data
Combining and processing
(78152, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'scores'] = scores[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

Renormalizing
Saving processed data


In [34]:
type_adata

AnnData object with n_obs × n_vars = 78152 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_2'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'le

In [36]:
len(out_files)

20

In [1]:
adata

NameError: name 'adata' is not defined

## Upload Cell Type data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [43]:
ss = hisepy.get_study_spaces()
study_space_uuid = ss[0]['id']
title = 'IDE for 05 PBMC L3 Pre-cleanup Unharmonized Clustering {d}'.format(d = date.today())
print(title)

IDE for 05 PBMC L3 Pre-cleanup Unharmonized Clustering 2024-10-14


In [40]:
search_id = element_id()
search_id

'osmium-rutherfordium-helium'

In [44]:
in_files = list(h5ad_uuids.values())
in_files

['7e87a3d2-3ac0-4b16-98e9-3d75753d128c',
 'cdcfd299-0f57-4b74-843a-fa57c157be15']

In [38]:
out_files

['output/up1_cluster_harmonize_ASDC_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_Activated_memory_B_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_Adaptive_NK_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_BaEoMaP_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_C1Qpos_CD16_monocyte_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD14pos_cDC2_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD27pos_effector_B_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD27neg_effector_B_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD4_MAIT_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD56bright_NK_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD8_MAIT_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD8aa_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CD95_memory_B_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CLP_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CM_CD4_T_cell_2024-08-21.h5ad',
 'output/up1_cluster_harmonize_CM_CD8_T

In [45]:
#getting the outfiles that I made a week ago
file_list=[]
path='/home/jupyter/certpro_up1/unharmonized_output/'
for file in os.listdir(path):
    file_list.append(path+file)

file_list.sort()
out_files = file_list[0:20]
out_files

['/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_ASDC_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Activated_memory_B_cell_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Adaptive_NK_cell_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_BaEoMaP_cell_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_C1Qpos_CD16_monocyte_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD27neg_effector_B_cell_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD27pos_effector_B_cell_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD4_MAIT_2024-09-30.h5ad',
 '/home/jupyter/certpro_up1/unharmonized

In [46]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination ='sodium-gadolinium-silicon'
)

you are trying to upload file_ids... ['/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_ASDC_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Activated_memory_B_cell_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Adaptive_NK_cell_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_BaEoMaP_cell_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_C1Qpos_CD16_monocyte_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD27neg_effector_B_cell_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD27pos_effector_B_cell_2024-09-30.h5ad', '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD4_MAIT_2024-09-30.h5ad', '/home/jupy

(y/n) y


{'trace_id': '594d1ad5-1703-44b8-9abc-3777ebac958c',
 'files': ['/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_ASDC_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Activated_memory_B_cell_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Adaptive_NK_cell_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_BaEoMaP_cell_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_C1Qpos_CD16_monocyte_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD27neg_effector_B_cell_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD27pos_effector_B_cell_2024-09-30.h5ad',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmon

In [38]:
import session_info
session_info.show()